In [ ]:
#conda activate base
#conda update pandas numpy

In [ ]:
import numpy as np
import pandas as pd

## Data_Preparation

In [2]:
df = pd.read_csv("C:\my files\projects\DA\Ops_Analytics_Project\data\PS_20174392719_1491204439457_log.csv") 

In [4]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

print("\nTransaction Types:")
print(df['type'].value_counts())

print("\nFirst 5 rows:")
print(df.head())
print(df.info())
print(df.describe())

# Check unique values
print("\n ")
print(df['type'].value_counts())
print(df['isFraud'].value_counts())

Shape: (6362620, 11)
Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

Transaction Types:
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

First 5 rows:
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        

## Feature_engineering

In [5]:
# === Create useful columns ===
df['is_success'] = 1 - df['isFraud']                    # Success flag
df['is_merchant'] = df['nameDest'].str.startswith('M')  # Merchants usually start with M

# Transaction category
df['transaction_category'] = df['type']

# === Aggregate at Merchant Level ===
merchant_stats = df.groupby('nameDest').agg(
    total_transactions=('amount', 'count'),
    total_volume=('amount', 'sum'),
    avg_amount=('amount', 'mean'),
    success_rate=('is_success', 'mean'),
    fraud_count=('isFraud', 'sum'),
    first_tx=('step', 'min'),
    last_tx=('step', 'max')
).reset_index()

merchant_stats['active_days'] = merchant_stats['last_tx'] - merchant_stats['first_tx']
merchant_stats['is_active'] = (merchant_stats['total_transactions'] >= 3).astype(int)

print("Total Unique Merchants:", merchant_stats.shape[0])
print(merchant_stats.head())

Total Unique Merchants: 2722362
      nameDest  total_transactions  total_volume     avg_amount  success_rate  \
0  C1000004082                   6    2259324.39  376554.065000           1.0   
1  C1000004940                  13    2534004.05  194923.388462           1.0   
2  C1000013769                  13    6204082.94  477237.149231           1.0   
3   C100001587                   9    1404313.66  156034.851111           1.0   
4  C1000015936                  16    2267960.19  141747.511875           1.0   

   fraud_count  first_tx  last_tx  active_days  is_active  
0            0       352      396           44          1  
1            0       131      377          246          1  
2            0       177      694          517          1  
3            0       138      310          172          1  
4            0        18      331          313          1  


## Cohort_and_KPIs

In [6]:
# === 1. Create Time Columns (Very Important) ===
df['hour'] = df['step'] % 24                    # Hour of the day (0-23)
df['day']   = (df['step'] // 24) % 30 + 1      # Day number (1 to 30)
df['month'] = (df['step'] // (24*30)) + 1      # Month number (1, 2, ...)

# === 2. Create Merchant Cohort (When the merchant first appeared) ===
# First, find the first transaction day for each merchant
merchant_first_day = df.groupby('nameDest')['day'].min().reset_index()
merchant_first_day = merchant_first_day.rename(columns={'day': 'cohort_day'}) # cohort_day = The first day a merchant appeared

# Merge this info back to main dataframe
df = df.merge(merchant_first_day, on='nameDest', how='left')

print(df[['step', 'day', 'month', 'cohort_day', 'is_success']].head()) 

   step  day  month  cohort_day  is_success
0     1    1      1           1           1
1     1    1      1           1           1
2     1    1      1           1           0
3     1    1      1           1           0
4     1    1      1           1           1


In [7]:
merchant_first_day.head()

,nameDest,cohort_day
0,C1000004082,15
1,C1000004940,6
2,C1000013769,8
3,C100001587,6
4,C1000015936,1


In [8]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,is_success,is_merchant,transaction_category,hour,day,month,cohort_day
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,1,True,PAYMENT,1,1,1,1
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,1,True,PAYMENT,1,1,1,1
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,0,False,TRANSFER,1,1,1,1
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,0,False,CASH_OUT,1,1,1,1
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,1,True,PAYMENT,1,1,1,1


## 04_KPIs_Calculation

In [9]:
print("=== Starting KPI Calculations ===\n")

df['day'] = (df['step'] // 24) % 30 + 1
df['month'] = (df['step'] // (24*30)) + 1
df['is_success'] = 1 - df['isFraud']

# Create Cohort (First transaction day per merchant)
merchant_cohort = df.groupby('nameDest')['day'].min().reset_index()
merchant_cohort = merchant_cohort.rename(columns={'day': 'cohort_day'})

# Remove the cohort_day column if it already exists from a previous run
if 'cohort_day' in df.columns:
    df = df.drop(columns=['cohort_day'])

df = df.merge(merchant_cohort, on='nameDest', how='left')


print("✅ Time columns + Cohort created\n")

=== Starting KPI Calculations ===

✅ Time columns + Cohort created



In [10]:
# ====================== OVERALL KPIs ======================
print("=== Overall Key Metrics ===")
print(f"Total Transactions     : {len(df):,}")
print(f"Overall Acceptance Rate: {df['is_success'].mean()*100:.2f} %")
print(f"Chargeback Rate        : {df['isFraud'].mean()*100:.4f} %")
print(f"Total Volume           : ${df['amount'].sum()/1_000_000:.1f} Million")

=== Overall Key Metrics ===
Total Transactions     : 6,362,620
Overall Acceptance Rate: 99.87 %
Chargeback Rate        : 0.1291 %
Total Volume           : $1144392.9 Million


In [36]:
# ====================== SAVE FILES FOR POWER BI ======================
output_path = r"C:\my files\projects\DA\Ops_Analytics_Project\data\\"

df.to_csv(output_path + "transactions_clean.csv", index=False)
print(f"✅ Saved: transactions_clean.csv ({len(df):,} rows)")

# Daily KPIs
daily_kpi = df.groupby('day').agg(
    total_transactions=('amount', 'count'),
    total_volume=('amount', 'sum'),
    acceptance_rate=('is_success', 'mean'),
    chargeback_rate=('isFraud', 'mean'),
    avg_amount=('amount', 'mean')
).round(4)

daily_kpi.to_csv(output_path + "daily_kpi.csv")
print("✅ Saved: daily_kpi.csv")

✅ Saved: transactions_clean.csv (6,362,620 rows)
✅ Saved: daily_kpi.csv


In [37]:
# Cohort KPIs
cohort_kpi = df.groupby('cohort_day').agg(
    num_merchants=('nameDest', 'nunique'),
    total_transactions=('amount', 'count'),
    total_volume=('amount', 'sum'),
    acceptance_rate=('is_success', 'mean'),
    chargeback_rate=('isFraud', 'mean')
).round(4)

cohort_kpi.to_csv(output_path + "cohort_kpi.csv")
print("✅ Saved: cohort_kpi.csv")

print("\n🎉 All files saved successfully! Ready for Power BI.")
print("\nFirst 5 rows of Daily KPI:")
print(daily_kpi.head())
print("\nFirst 5 rows of Cohort KPI:")
print(cohort_kpi.head())

✅ Saved: cohort_kpi.csv

🎉 All files saved successfully! Ready for Power BI.

First 5 rows of Daily KPI:
     total_transactions  total_volume  acceptance_rate  chargeback_rate  \
day                                                                       
1                571321  9.218941e+10           0.9990           0.0010   
2                452761  7.108987e+10           0.9993           0.0007   
3                  6749  9.281746e+08           0.9547           0.0453   
4                 21904  3.151848e+09           0.9876           0.0124   
5                 12995  1.686137e+09           0.9812           0.0188   

      avg_amount  
day               
1    161361.8370  
2    157014.1132  
3    137527.7274  
4    143893.7148  
5    129752.7548  

First 5 rows of Cohort KPI:
            num_merchants  total_transactions  total_volume  acceptance_rate  \
cohort_day                                                                     
1                  247979             1546524  